# Long Context for Video Generation

The problem: 10s video at 24fps = 240 frames. At 64x64 latent per frame = ~1M tokens. Full self-attention is O(N²) in memory and compute. A 2-minute video would need ~3M tokens -- impossible with standard attention.

This is an active research problem that can mean many things: sparsity, linear attention variants, context compression. There's no single solution -- understand the design space.

**Time:** ~2-3 hrs, needs GPU for demos.

In [ ]:
# EXERCISE: Implement sliding window attention

def sliding_window_attention(
    q: torch.Tensor,
    k: torch.Tensor,
    v: torch.Tensor,
    window_size: int,
) -> torch.Tensor:
    """Sliding window attention: each position attends to +-window_size neighbors.

    Complexity: O(N * window_size) instead of O(N^2).

    Args:
        q, k, v: (B, N, D) query, key, value tensors
        window_size: number of positions to attend in each direction

    Returns:
        (B, N, D) attention output
    """
    # YOUR CODE
    raise NotImplementedError


# Test
B, N, D = 2, 32, 64
q = k = v = torch.randn(B, N, D)
out = sliding_window_attention(q, k, v, window_size=4)
assert out.shape == (B, N, D)
print(f"Sliding window attention: {q.shape} -> {out.shape}")


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import time
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Demonstrate O(N^2) scaling of attention
seq_lengths = [256, 512, 1024, 2048, 4096, 8192]
memory_usage = []
compute_time = []

for n in tqdm(seq_lengths, desc="Measuring attention scaling"):
    d = 64  # head dim
    q = torch.randn(1, 8, n, d, device=device)  # 8 heads
    k = torch.randn(1, 8, n, d, device=device)
    v = torch.randn(1, 8, n, d, device=device)

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    start = time.perf_counter()

    # Standard attention
    attn = torch.matmul(q, k.transpose(-2, -1)) / (d ** 0.5)
    attn = F.softmax(attn, dim=-1)
    out = torch.matmul(attn, v)

    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    if device.type == "cuda":
        peak_mem = torch.cuda.max_memory_allocated() / 1e9
    else:
        # Estimate: Q,K,V + attn matrix + output
        peak_mem = (3 * 1 * 8 * n * d + 1 * 8 * n * n + 1 * 8 * n * d) * 4 / 1e9

    memory_usage.append(peak_mem)
    compute_time.append(elapsed)

    del q, k, v, attn, out
    if device.type == "cuda":
        torch.cuda.empty_cache()

# Plot scaling
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(seq_lengths, memory_usage, 'bo-', label='Measured')
axes[0].set_xlabel("Sequence Length")
axes[0].set_ylabel("Peak GPU Memory (GB)")
axes[0].set_title("Attention Memory: O(N²)")
axes[0].set_xscale('log', base=2)

# Overlay theoretical O(N^2) curve
theoretical = [memory_usage[0] * (n / seq_lengths[0]) ** 2 for n in seq_lengths]
axes[0].plot(seq_lengths, theoretical, 'r--', label='O(N²) theoretical')
axes[0].legend()

axes[1].plot(seq_lengths, compute_time, 'go-', label='Measured')
axes[1].set_xlabel("Sequence Length")
axes[1].set_ylabel("Compute Time (s)")
axes[1].set_title("Attention Compute: O(N²)")
axes[1].set_xscale('log', base=2)

theoretical_t = [compute_time[0] * (n / seq_lengths[0]) ** 2 for n in seq_lengths]
axes[1].plot(seq_lengths, theoretical_t, 'r--', label='O(N²) theoretical')
axes[1].legend()

plt.tight_layout()
plt.show()

# Print the numbers
print("\nVideo context sizes:")
print(f"  1s @ 24fps, 64x64 latent: {24 * 64 * 64:,} tokens")
print(f"  10s @ 24fps, 64x64 latent: {240 * 64 * 64:,} tokens")
print(f"  2min @ 24fps, 64x64 latent: {2880 * 64 * 64:,} tokens")
print(f"\nAttention matrix for 2min video: {2880 * 4096}² = {(2880 * 4096) ** 2:.2e} elements")
print("This is why we need alternatives to full attention.")

In [ ]:
def sliding_window_attention(q, k, v, window_size=256):
    """Sliding window attention: each token attends to ±window_size neighbors."""
    B, H, N, D = q.shape
    out = torch.zeros_like(v)

    for i in range(N):
        start = max(0, i - window_size)
        end = min(N, i + window_size + 1)

        qi = q[:, :, i:i + 1]           # [B, H, 1, D]
        ki = k[:, :, start:end]          # [B, H, window, D]
        vi = v[:, :, start:end]

        attn = torch.matmul(qi, ki.transpose(-2, -1)) / (D ** 0.5)
        attn = F.softmax(attn, dim=-1)
        out[:, :, i:i + 1] = torch.matmul(attn, vi)

    return out

# Visualize the attention pattern
n = 32  # small for visualization
window = 8

# Build the attention mask
mask = torch.zeros(n, n)
for i in range(n):
    start = max(0, i - window)
    end = min(n, i + window + 1)
    mask[i, start:end] = 1

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Full attention
axes[0].imshow(torch.ones(n, n), cmap='Blues', vmin=0, vmax=1)
axes[0].set_title(f"Full Attention\nO(N²) = {n * n} ops")
axes[0].set_xlabel("Key")
axes[0].set_ylabel("Query")

# Sliding window
axes[1].imshow(mask, cmap='Blues', vmin=0, vmax=1)
axes[1].set_title(f"Sliding Window (w={window})\nO(N·w) = {int(mask.sum())} ops")
axes[1].set_xlabel("Key")
axes[1].set_ylabel("Query")

# Causal sliding window (for AR models)
causal_mask = mask * torch.tril(torch.ones(n, n))
axes[2].imshow(causal_mask, cmap='Blues', vmin=0, vmax=1)
axes[2].set_title(f"Causal Sliding Window\nO(N·w/2) = {int(causal_mask.sum())} ops")
axes[2].set_xlabel("Key")
axes[2].set_ylabel("Query")

plt.tight_layout()
plt.show()

print(f"Computation savings: sliding window uses {mask.sum() / n / n * 100:.1f}% of full attention")

In [ ]:
# STA extends sliding window to 3D (height, width, time)
# Instead of 1D window, use a 3D tile that slides over the spatiotemporal volume

def visualize_sta_pattern(n_frames=8, spatial=4):
    """Visualize STA attention pattern for video."""
    # Total tokens = n_frames * spatial * spatial
    N = n_frames * spatial * spatial

    # Full attention pattern
    full_mask = np.ones((N, N))

    # STA pattern: each token attends to its local 3D tile
    # Tile size: 2 frames x 2x2 spatial = 8 tokens (for illustration)
    tile_t, tile_h, tile_w = 2, 2, 2
    sta_mask = np.zeros((N, N))

    for qi in range(N):
        # Decompose query token into (t, h, w)
        qt = qi // (spatial * spatial)
        qr = qi % (spatial * spatial)
        qh = qr // spatial
        qw = qr % spatial

        # Tile neighborhood
        for kt in range(max(0, qt - tile_t), min(n_frames, qt + tile_t + 1)):
            for kh in range(max(0, qh - tile_h), min(spatial, qh + tile_h + 1)):
                for kw in range(max(0, qw - tile_w), min(spatial, qw + tile_w + 1)):
                    ki = kt * spatial * spatial + kh * spatial + kw
                    sta_mask[qi, ki] = 1

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    axes[0].imshow(full_mask, cmap='Blues', aspect='auto')
    axes[0].set_title(f"Full 3D Attention\n{N}x{N} = {N * N:,} elements")

    axes[1].imshow(sta_mask, cmap='Blues', aspect='auto')
    sparsity = sta_mask.sum() / (N * N) * 100
    axes[1].set_title(f"Sliding Tile Attention (STA)\n{sparsity:.1f}% density ({sta_mask.sum():.0f}/{N * N:,} elements)")

    for ax in axes:
        # Add frame boundaries
        for f in range(1, n_frames):
            pos = f * spatial * spatial
            ax.axhline(pos - 0.5, color='red', linewidth=0.5, alpha=0.5)
            ax.axvline(pos - 0.5, color='red', linewidth=0.5, alpha=0.5)
        ax.set_xlabel("Key token")
        ax.set_ylabel("Query token")

    plt.suptitle(f"STA: {n_frames} frames, {spatial}x{spatial} spatial, tile={tile_t}x{tile_h}x{tile_w}")
    plt.tight_layout()
    plt.show()

    return sparsity

sparsity = visualize_sta_pattern()
print(f"\nSTA uses {sparsity:.1f}% of full attention computation")
print("For real video (e.g., 240 frames x 64x64), STA is essential")
print("Paper: Sliding Tile Attention (arxiv 2502.04507)")

## Linear Attention Variants

Goal: O(N) complexity instead of O(N²). The key insight: standard attention computes `softmax(QK^T/√d)V`, which requires materializing the N×N attention matrix. Linear attention avoids this.

**Kernelized Attention:** Replace `softmax(QK^T)` with `φ(Q)φ(K)^T` where φ is a feature map. Then compute `φ(Q)(φ(K)^TV)` -- the inner product is d×d instead of N×N. Cost: O(N·d²) instead of O(N²·d).

**Causal Linear Attention:** For AR models, maintain a running state `S_t = Σ_{i≤t} φ(k_i)v_i^T`. At each step: `output_t = φ(q_t)S_t`. Constant time per step, like an RNN! This is the connection to state-space models.

**The tradeoff:** Linear attention approximates softmax attention. The approximation quality varies -- some tasks degrade significantly. This is why it's a research problem, not a solved problem.

## State-Space Models for Video

SSMs (Mamba, S4) are an alternative to attention that naturally handles long sequences.

**Mamba (arxiv 2312.00752):**
- Selective state-space model: input-dependent state transitions
- O(N) computation and memory -- linear in sequence length
- Processes sequences like an RNN but trains like a CNN (parallel scan)
- No attention matrix at all -- information flows through a compressed state

**"Long-Context State-Space Video World Models" (ICCV 2025):**
- Hybrid SSM + local causal attention
- SSM handles long-range temporal memory (what happened 100 frames ago)
- Local causal attention handles fine-grained spatial details (within a few frames)
- Constant per-frame inference speed regardless of context length -- the key advantage for long video

**M4V: Mamba for Text-to-Video:**
- Replace temporal attention with Mamba blocks
- 45% fewer FLOPs vs attention at 768x1280 resolution
- Comparable quality on standard benchmarks

In [ ]:
def measure_attention_memory(attn_fn, seq_lengths, d_model=64, n_heads=8):
    """Measure peak GPU memory for different attention mechanisms."""
    results = []
    d_head = d_model // n_heads

    for n in seq_lengths:
        try:
            q = torch.randn(1, n_heads, n, d_head, device=device)
            k = torch.randn(1, n_heads, n, d_head, device=device)
            v = torch.randn(1, n_heads, n, d_head, device=device)

            if device.type == "cuda":
                torch.cuda.reset_peak_memory_stats()
                torch.cuda.synchronize()

            start = time.perf_counter()
            out = attn_fn(q, k, v)

            if device.type == "cuda":
                torch.cuda.synchronize()
            elapsed = time.perf_counter() - start

            if device.type == "cuda":
                mem = torch.cuda.max_memory_allocated() / 1e9
            else:
                # Rough estimate for CPU
                mem = (3 * 1 * n_heads * n * d_head + 1 * n_heads * n * n + 1 * n_heads * n * d_head) * 4 / 1e9

            results.append({"n": n, "memory_gb": mem, "time_s": elapsed, "oom": False})

            del q, k, v, out
            if device.type == "cuda":
                torch.cuda.empty_cache()
        except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
            if "out of memory" in str(e).lower():
                results.append({"n": n, "memory_gb": float('nan'), "time_s": float('nan'), "oom": True})
                if device.type == "cuda":
                    torch.cuda.empty_cache()
            else:
                raise

    return results


def full_attention(q, k, v):
    """Standard O(N²) attention."""
    attn = torch.matmul(q, k.transpose(-2, -1)) / (q.shape[-1] ** 0.5)
    return torch.matmul(F.softmax(attn, dim=-1), v)


def window_attention(q, k, v, window=512):
    """Efficient sliding window via chunked attention."""
    B, H, N, D = q.shape
    out = torch.zeros_like(v)
    for start in range(0, N, window):
        end = min(start + window, N)
        ctx_start = max(0, start - window)
        qi = q[:, :, start:end]
        ki = k[:, :, ctx_start:end]
        vi = v[:, :, ctx_start:end]
        attn = torch.matmul(qi, ki.transpose(-2, -1)) / (D ** 0.5)
        out[:, :, start:end] = torch.matmul(F.softmax(attn, dim=-1), vi)
    return out


def linear_attention(q, k, v):
    """Linear attention: phi(Q)(phi(K)^T V) where phi = elu + 1."""
    phi_q = F.elu(q) + 1
    phi_k = F.elu(k) + 1
    kv = torch.matmul(phi_k.transpose(-2, -1), v)  # [B, H, D, D]
    denom = torch.matmul(
        phi_q, phi_k.transpose(-2, -1).sum(-1, keepdim=True)
    ) + 1e-6
    return torch.matmul(phi_q, kv) / denom


test_lengths = [1024, 2048, 4096, 8192, 16384]

print("Measuring full attention...")
full_results = measure_attention_memory(full_attention, test_lengths)
print("Measuring sliding window attention...")
window_results = measure_attention_memory(window_attention, test_lengths)
print("Measuring linear attention...")
linear_results = measure_attention_memory(linear_attention, test_lengths)

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for results, label, color in [
    (full_results, "Full Attention O(N²)", "red"),
    (window_results, "Sliding Window O(N·w)", "blue"),
    (linear_results, "Linear Attention O(N)", "green"),
]:
    valid = [r for r in results if not r["oom"]]
    if valid:
        axes[0].plot(
            [r["n"] for r in valid],
            [r["memory_gb"] for r in valid],
            'o-', label=label, color=color,
        )
        axes[1].plot(
            [r["n"] for r in valid],
            [r["time_s"] for r in valid],
            'o-', label=label, color=color,
        )

axes[0].set_xlabel("Sequence Length")
axes[0].set_ylabel("Memory (GB)")
axes[0].set_title("Memory Scaling")
axes[0].legend()
axes[0].set_xscale('log', base=2)

axes[1].set_xlabel("Sequence Length")
axes[1].set_ylabel("Time (s)")
axes[1].set_title("Compute Scaling")
axes[1].legend()
axes[1].set_xscale('log', base=2)

plt.tight_layout()
plt.show()

# Print results table
print("\n{:<10} {:>12} {:>12} {:>12} {:>12} {:>12} {:>12}".format(
    "SeqLen", "Full Mem", "Full Time", "Window Mem", "Window Time", "Linear Mem", "Linear Time"
))
print("-" * 82)
for i, n in enumerate(test_lengths):
    fm = f"{full_results[i]['memory_gb']:.3f}" if not full_results[i]['oom'] else "OOM"
    ft = f"{full_results[i]['time_s']:.4f}" if not full_results[i]['oom'] else "OOM"
    wm = f"{window_results[i]['memory_gb']:.3f}" if not window_results[i]['oom'] else "OOM"
    wt = f"{window_results[i]['time_s']:.4f}" if not window_results[i]['oom'] else "OOM"
    lm = f"{linear_results[i]['memory_gb']:.3f}" if not linear_results[i]['oom'] else "OOM"
    lt = f"{linear_results[i]['time_s']:.4f}" if not linear_results[i]['oom'] else "OOM"
    print(f"{n:<10} {fm:>12} {ft:>12} {wm:>12} {wt:>12} {lm:>12} {lt:>12}")

## Context Compression

Alternative approach: instead of making attention cheaper, make the sequence shorter.

**Token Merging (ToMe):** Progressively merge similar tokens during generation. Fewer tokens = less attention compute. Works well for spatial redundancy (nearby patches are similar).

**Latent Summarization:** Compress past context into fixed-size "summary" tokens. Like a memory bank. The model attends to summaries of old frames + full representations of recent frames.

**Memory Banks:** Maintain a fixed-size buffer of key-value pairs from past frames. When buffer is full, merge or evict oldest entries. Constant memory regardless of video length.

The tradeoff with all compression: information loss. Long-range details (a character that appeared 30 seconds ago) may be forgotten. The research question: what to compress and what to keep?

## Checkpoint

How do you generate a 2-minute video without running out of memory?

Approach depends on the architecture:

1. **If using diffusion (all frames at once):** STA/sliding tile attention to reduce O(N²) to O(N·tile_size). Combined with temporal chunking (generate 4-second segments with overlapping context).
2. **If using AR (frame by frame):** Linear attention or SSM for long-range context (O(N) per frame). Sliding window for local detail. Memory bank for events from minutes ago.
3. **Hybrid:** SSM backbone for temporal (constant cost per frame, arbitrary context) + local attention for spatial (high quality within each frame). This is the approach from "Long-Context State-Space Video World Models."
4. **Practical hack:** Multi-resolution -- generate at low res with full context, then super-resolve local segments.

The honest answer: nobody has solved this well for high-quality long video. This remains active research because the problem is genuinely open.

Further reading: STA (2502.04507), Mamba (2312.00752), S4 (2111.00396), ToMe (2210.09461), M4V.